PRODUCER SCRAPER

Import

In [5]:
import json
import re

import csv
import os
import threading
import queue
import time

#Data Cleaning
import bs4 as bs 
import requests as req
import pandas as pd
import numpy as np

#Selenium
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup



Settings and Constants


In [11]:
#browser and driver options 
options = Options()
options.binary_location = "C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe"



fresh_href = r"C:\Users\user\Documents\ProgrammingHell\Python\TurboAz scrapper\turboaz_scraper_2026\hrefs\fresh_href.csv"

#websites = []
max_pages = 40
page_count = 0

Functions

In [80]:

  
def pop_fresh_href():
    """Take the first href off the fresh queue, return it (or None if empty)."""
    if not os.path.exists(fresh_href):
        return None
    with open(fresh_href, newline='', encoding='utf-8') as f:
        rows = list(csv.reader(f))
    if not rows:
        return None
    href, rest = rows[0][0], rows[1:]
    with open(fresh_href, 'w', newline='', encoding='utf-8') as f:
        csv.writer(f).writerows(rest)
    return href



class Scraper:
    def __init__(self,num_consumers=2 ):
        
                #--- Constants
        self.main_page = "https://turbo.az"
        self.time_quota = 4
        self.num_consumers = num_consumers
        self.websites = self.producer_link_gen(4)

                #--- Shared state across threads
        self.url_queue = queue.Queue()      # thread-safe by design, no lock needed
        self.seen_urls = set()              # NOT thread-safe on its own
        self.seen_lock = threading.Lock()   # protects self.seen_urls

        self.results = []                   # where scraped data ends up
        self.results_lock = threading.Lock()
        
    def create_driver(self):
        '''
        This function supposed to be called by individual threads and creates webdriver instances.
        (Using global driver is not Thread safe!!!)
        '''
        
        service = Service("C:\\chromedriver-win32\\chromedriver.exe")
        options = webdriver.ChromeOptions()
        options.binary_location = "C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe"
        # options.add_argument('--headless')
        # options.add_argument('--no-sandbox')
        # options.add_argument('--disable-dev-shm-usage')
        
        return webdriver.Chrome(options=options, service=service)
    
    


    def crawler(self, url, driver) -> list:
        '''
        Crawler scraper. Scrapes hrefs,
        creats full link connecting href with
        main page(turbo.az) and sends link to the producer
        '''
        driver.get(url)
        soup = BeautifulSoup(driver.page_source)
    
        return [
            f"{self.main_page}{i['href']}"
            for i in soup.find_all('a', href=True)
            if re.fullmatch(r'/autos/\d+[\w-]*', i['href'])
        ]
    def producer_link_gen(self, max_pages):
        for page_num in range(1, max_pages + 1):
            yield f'https://turbo.az/autos?page={page_num}'
    def producer(self, url):
        """
        Initializes Crawler and redirects links to consumer
        """
        driver = self.create_driver()
        try:
            hrefs = self.crawler(url, driver)
            print(hrefs)
            print(len(hrefs))
            for href in hrefs:
                with self.seen_lock:
                    if href in self.seen_urls:
                        continue
                    self.seen_urls.add(href)
                self.url_queue.put(href)
            print('crawler started')    
        finally:
            driver.quit()

    #--- consumers and misc
    def fetch_details(self, href, driver):
        """
        Takes parameters from consumer and scrapes whatever i needed from targeted pages(Test)
        """
        driver.get(href)
        soup = BeautifulSoup(driver.page_source, "html.parser")
        details = soup.select(
            '.product-properties.tz-d-flex.tz-justify-between.tz-gap-10'
        )
        return details
    
    def consumer(self):
        driver = self.create_driver()

        try:
            while True:
                href = self.url_queue
                if href is None:
                    break
                data = self.fetch_details(href, driver)
                with self.results_lock:
                    self.results.append(data)
                self.url_queue.task_done()
                print('consumer started')
        finally:
            webdriver.quit()

        

    def run(self):
        # 1. Start consumers FIRST — they'll just block on an empty
        #    queue until producers put work in.
        consumer_threads = [
            threading.Thread(target=self.consumer)
            for _ in range(self.num_consumers)
        ]
        for t in consumer_threads:
            t.start()

        
        # 2. Start one producer(Crawler starter) thread per website.
        producer_threads = [
            threading.Thread(target=self.producer, args=(url,))
            for url in self.websites
        ]
        for t in producer_threads:
            t.start()
        for t in producer_threads:
            t.join()   # wait until all producers finish finding hrefs



        # 3. Now that no more hrefs are coming, tell each consumer to stop
        for _ in range(self.num_consumers):
            self.url_queue.put(None)

        for t in consumer_threads:
            t.join()

        return self.results        

def main():
    print("Starting web scraping with multithreading...")
    start_time = time.time()
    scraper = Scraper(
        num_consumers=2
    )
    results = scraper.run()

    end_time = time.time()
    print(f"Scraped {len(results)} listings in {end_time - start_time:.2f} seconds")

  

Scraper 

In [81]:
if __name__ == "__main__":
    main()

Starting web scraping with multithreading...


Exception in thread Thread-586 (consumer):
Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Temp\ipykernel_13856\503277343.py", line 107, in consumer
    data = self.fetch_details(href, driver)
  File "C:\Users\user\AppData\Local\Temp\ipykernel_13856\503277343.py", line 92, in fetch_details
    driver.get(href)
    ~~~~~~~~~~^^^^^^
  File "c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\selenium\webdriver\remote\webdriver.py", line 512, in get
    self.execute(Command.GET, {"url": url})
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\selenium\webdriver\remote\webdriver.py", line 489, in execute
    response = cast(RemoteConnection, self.command_executor).execute(driver_command, params)
  File "c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\selenium\webdriver\remote\remote_connection.py", line 403, in execute
    data = utils.dump_json(params

[]
0
crawler started
['https://turbo.az/autos/9553808-byd-qin-plus', 'https://turbo.az/autos/10266177-changan-oshan-x5-plus', 'https://turbo.az/autos/10396758-jaecoo-j8', 'https://turbo.az/autos/10482382-kia-optima', 'https://turbo.az/autos/10165048-ferrari-296-gtb', 'https://turbo.az/autos/10208068-hyundai-elantra', 'https://turbo.az/autos/10487912-bmw-530', 'https://turbo.az/autos/10494370-ford-transit', 'https://turbo.az/autos/10500863-bmw-320d', 'https://turbo.az/autos/10508895-bmw-523', 'https://turbo.az/autos/10510852-byd-destroyer-05', 'https://turbo.az/autos/10512907-ford-fusion-north-america', 'https://turbo.az/autos/10248087-bmw-528', 'https://turbo.az/autos/10278836-land-rover-rr-sport', 'https://turbo.az/autos/10416376-avatr-12', 'https://turbo.az/autos/10517120-lada-vaz-2109', 'https://turbo.az/autos/10517129-hyundai-santa-fe', 'https://turbo.az/autos/10501417-volkswagen-touareg', 'https://turbo.az/autos/10443231-bmw-525', 'https://turbo.az/autos/10517125-toyota-aqua', 'ht

Buffer +

Buffer -

Examples and Ideas

In [55]:
def crawler_link_gen( max_pages=10):
    for page_num in range(1, max_pages + 1):
        yield f'https://turbo.az/autos?page={page_num}'

print(list(crawler_link_gen(20)))

['https://turbo.az/autos?page=1', 'https://turbo.az/autos?page=2', 'https://turbo.az/autos?page=3', 'https://turbo.az/autos?page=4', 'https://turbo.az/autos?page=5', 'https://turbo.az/autos?page=6', 'https://turbo.az/autos?page=7', 'https://turbo.az/autos?page=8', 'https://turbo.az/autos?page=9', 'https://turbo.az/autos?page=10', 'https://turbo.az/autos?page=11', 'https://turbo.az/autos?page=12', 'https://turbo.az/autos?page=13', 'https://turbo.az/autos?page=14', 'https://turbo.az/autos?page=15', 'https://turbo.az/autos?page=16', 'https://turbo.az/autos?page=17', 'https://turbo.az/autos?page=18', 'https://turbo.az/autos?page=19', 'https://turbo.az/autos?page=20']
